# Arc — CUDA build check on a free GPU (Google Colab)

**What this does:** clones Arc at a chosen commit and compiles every Arc-authored
CUDA kernel (QTIP + arc-cuda-graph) using Colab's free `nvcc`. Compiling CUDA needs
only the toolkit, *not* a matching GPU — so we cross-compile for **sm_90 (H100/Hopper)**,
the V4 Flash rental target, regardless of which GPU Colab hands you.

**Honest scope / limits:**
- This is the same gate as `.github/workflows/cuda_compile_check.yaml` and
  `arc-tools/cuda_compile_check.sh`. It proves the kernels **compile** for the rental arch.
- Free Colab/Kaggle GPUs are **T4 (sm_75)**. The QTIP kernels are gated to **sm_80+**
  (`has_qtip_kernels` in `mistralrs-quant/build.rs`), so a free GPU **cannot run** them.
  Runtime kernel validation (the parity tests) needs an sm_80+ box — i.e. the paid rental,
  or a Colab Pro A100. This notebook will auto-run the GPU smoke tests only if it detects sm_80+.
- Use **GitHub Actions** (free, automatic, no GPU) as the primary gate; this notebook is a
  second free path that uses a real `nvcc` and, on sm_80+ hardware, real kernel execution.

Set `Runtime -> Change runtime type -> GPU` before running (gives you nvcc + a GPU).

## 1. Inspect the GPU + CUDA toolkit Colab gave you

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv || echo 'no GPU — compile-only still works'
!nvcc --version

## 2. Choose the commit to validate
Defaults to `master`. Pin to a SHA to validate exactly what the rental will check out.

In [ ]:
ARC_REPO = 'https://github.com/aeonmindai/arc.git'
ARC_REF  = 'master'   # e.g. '12527af2d' to pin an exact commit
print('repo:', ARC_REPO, '\nref :', ARC_REF)

## 3. Install Rust (stable)

In [ ]:
import os
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!cargo --version

## 4. Clone Arc at the chosen ref

In [ ]:
import os
!rm -rf /content/arc
!git clone --filter=blob:none $ARC_REPO /content/arc
!cd /content/arc && git checkout $ARC_REF && git log -1 --oneline
os.chdir('/content/arc')

## 5. Run the CUDA compile gate (sm_90)
Uses `arc-tools/cuda_compile_check.sh`. `FEATURES=cuda` (no flash-attn) keeps the build
within Colab's RAM/disk. `CUDA_COMPUTE_CAP=90` cross-compiles the QTIP kernels for Hopper
even on a T4. If Colab gave you an sm_80+ GPU (A100 on Pro), the script also runs the
QTIP GPU parity tests automatically.

In [ ]:
!cd /content/arc && \
  CUDA_COMPUTE_CAP=90 FEATURES=cuda RUN_GPU_TESTS=auto \
  bash arc-tools/cuda_compile_check.sh

## 6. (Optional) Force the QTIP GPU parity tests
Only meaningful on an **sm_80+** GPU. On a T4 this will report that the QTIP kernels
are not compiled in for sm_75 — that is expected, not a failure of Arc.

In [ ]:
# Detect arch and run real kernel tests only if >= sm_80.
import subprocess
cc = subprocess.run(['bash','-lc',
   "nvidia-smi --query-gpu=compute_cap --format=csv,noheader 2>/dev/null | head -1 | tr -d '. '"],
   capture_output=True, text=True).stdout.strip() or '0'
print('compute_cap =', cc)
if cc.isdigit() and int(cc) >= 80:
    !cd /content/arc && CUDA_COMPUTE_CAP=$cc cargo test --release -p mistralrs-quant --features cuda -- --nocapture --test-threads=1 cuda_quantize_matches_cpu_dequantize_cos_sim cuda_fused_gemv_matches_dequant_matmul qtip_gather_forward_cuda_matches_cpu
else:
    print('GPU is sm_%s (< sm_80) — QTIP kernels are not enabled here.' % cc)
    print('Compile gate above already proved the kernels build for sm_90 (the rental arch).')
    print('Runtime parity must be validated on the rental or a Colab Pro A100.')